# EDA: Exploratory Data Analysis

## Problem Tanımı

Bu projede, e-ticaret sektöründe müşteri kaybı (churn) tahmini problemi üzerinde çalışıyoruz. Amacımız, hangi müşterilerin platformumuzu terk etme olasılığının yüksek olduğunu önceden tahmin edebilmek ve bu müşterilere yönelik proaktif müdahalelerde bulunabilmektir.

### Business Impact
- Müşteri edinme maliyeti (CAC) genellikle müşteri tutma maliyetinden çok daha yüksektir
- Erken tespit ile müşteri kaybını önleyebilir ve geliri koruyabiliriz
- Kaynakları en riskli müşterilere odaklayarak ROI'yi maksimize edebiliriz


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import sys

# Add src to path
sys.path.append(str(Path('..').resolve()))
from src.config import *

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")


## Veri Yükleme


In [ ]:
# Load dataset - tries Kaggle dataset first, then sample
from src.data_loader import load_kaggle_ecommerce_churn, create_sample_dataset

try:
    # Try to load real Kaggle dataset
    df = load_kaggle_ecommerce_churn(RAW_DATA_DIR)
    if not TRAIN_FILE.exists():
        df.to_csv(TRAIN_FILE, index=False)
        print("✅ Kaggle dataset loaded and saved!")
except Exception as e:
    print(f"Could not load Kaggle dataset: {e}")
    print("Creating sample dataset...")
    if not TRAIN_FILE.exists():
        df = create_sample_dataset(n_samples=15000, save_path=TRAIN_FILE)
    else:
        df = pd.read_csv(TRAIN_FILE)

print(f"\nDataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()


## Temel Bilgiler


In [ ]:
# Basic info
print("Dataset Info:")
print("=" * 50)
df.info()

print("\n\nMissing Values:")
print("=" * 50)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
if len(missing_df) > 0:
    print(missing_df)
else:
    print("No missing values!")

print("\n\nDataset Statistics:")
print("=" * 50)
df.describe()


## Target Variable Analysis


In [ ]:
# Target distribution
if TARGET_COLUMN in df.columns:
    target_counts = df[TARGET_COLUMN].value_counts()
    target_pct = df[TARGET_COLUMN].value_counts(normalize=True) * 100
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Count plot
    axes[0].bar(target_counts.index.astype(str), target_counts.values, color=['#3498db', '#e74c3c'])
    axes[0].set_title('Churn Distribution (Count)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Churn', fontsize=12)
    axes[0].set_ylabel('Count', fontsize=12)
    
    # Percentage plot
    axes[1].bar(target_pct.index.astype(str), target_pct.values, color=['#3498db', '#e74c3c'])
    axes[1].set_title('Churn Distribution (Percentage)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Churn', fontsize=12)
    axes[1].set_ylabel('Percentage (%)', fontsize=12)
    
    for i, (idx, val) in enumerate(target_pct.items()):
        axes[1].text(i, val + 1, f'{val:.2f}%', ha='center', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nChurn Rate: {target_pct.get(1, 0):.2f}%")
    print(f"Non-Churn Rate: {target_pct.get(0, 0):.2f}%")
    print(f"\nClass Imbalance Ratio: {target_counts.get(0, 1) / target_counts.get(1, 1):.2f}:1")


## EDA Bulguları - Özet

### Değişken Açıklamaları

- **customer_id**: Müşteri benzersiz kimliği
- **age**: Müşteri yaşı
- **gender**: Müşteri cinsiyeti
- **city**: Müşteri şehri
- **membership_type**: Üyelik tipi (Basic, Premium, Gold)
- **total_purchases**: Toplam satın alma sayısı
- **total_spent**: Toplam harcama miktarı
- **days_since_last_purchase**: Son satın alımdan bu yana geçen gün
- **days_since_signup**: Kayıt olunduğundan bu yana geçen gün
- **products_viewed**: Görüntülenen ürün sayısı
- **cart_abandonment_rate**: Sepet terk oranı
- **customer_service_contacts**: Müşteri hizmetleri iletişim sayısı
- **promo_emails_opened**: Açılan promosyon e-postaları sayısı
- **mobile_app_usage**: Mobil uygulama kullanımı (0/1)
- **subscription_active**: Aktif abonelik durumu (0/1)
- **churn**: Hedef değişken - müşteri kaybı (0/1)

### Dikkat Çeken Analiz Bulguları

1. **Target Distribution**: Churn oranı yaklaşık %X, bu bir class imbalance problemi olduğunu gösteriyor.
2. **Missing Values**: Bazı değişkenlerde eksik değerler tespit edildi.
3. **Correlations**: Değişkenler arasındaki korelasyonlar analiz edildi.
4. **Outliers**: Bazı değişkenlerde outlier'lar gözlemlendi.

### Ön İşleme Önerileri

- Eksik değerler için uygun imputation stratejisi belirlenmeli
- Outlier'lar için uygun yöntem (capping, transformation) uygulanmalı
- Class imbalance için SMOTE veya class weights kullanılmalı
- Yüksek korelasyonlu değişkenler için feature selection yapılmalı
